In [ ]:
# Example 1: Depends with a sync factory + resolved_dependencies.
from uncalled_for import Depends, resolved_dependencies

def get_config() -> dict:
    return {"env": "test", "debug": True}

async def handler(config: dict = Depends(get_config)) -> str:
    return f"env={config['env']}"

async with resolved_dependencies(handler) as resolved:
    print(f"resolved keys: {sorted(resolved.keys())}")
    result = await handler(**resolved)
    print(f"handler result: {result}")
    assert result == "env=test"
    assert resolved["config"] == {"env": "test", "debug": True}

In [ ]:
# Example 2: async context-manager factory gets enter/exit lifecycle.
# (Factories must return a context manager for lifecycle management —
# wrap async generators with contextlib.asynccontextmanager.)
from contextlib import asynccontextmanager

from uncalled_for import Depends, resolved_dependencies

events: list[str] = []

@asynccontextmanager
async def get_resource():
    events.append("open")
    try:
        yield {"handle": 42}
    finally:
        events.append("close")

async def use_resource(res: dict = Depends(get_resource)) -> int:
    return res["handle"]

async with resolved_dependencies(use_resource) as resolved:
    value = await use_resource(**resolved)
    assert value == 42

assert events == ["open", "close"], events
print(f"lifecycle events: {events}")

In [ ]:
# Example 3: introspection + without_dependencies wrapping.
import inspect

from uncalled_for import Depends, get_dependency_parameters, without_dependencies

def factory_a() -> int:
    return 1

def factory_b() -> str:
    return "hello"

async def my_func(x: int, a: int = Depends(factory_a), b: str = Depends(factory_b)) -> str:
    return f"{x}/{a}/{b}"

assert set(get_dependency_parameters(my_func)) == {"a", "b"}
wrapped = without_dependencies(my_func)
assert list(inspect.signature(wrapped).parameters) == ["x"]
result = await wrapped(x=7)
assert result == "7/1/hello", result
print(f"without_dependencies result: {result}")

In [ ]:
# Example 4: Shared + SharedContext caches across calls.
from uncalled_for import Shared, SharedContext, resolved_dependencies

init_count = 0

def make_app() -> dict:
    global init_count
    init_count += 1
    return {"id": init_count}

async def handler1(app: dict = Shared(make_app)) -> int:
    return app["id"]

async def handler2(app: dict = Shared(make_app)) -> int:
    return app["id"]

async with SharedContext():
    async with resolved_dependencies(handler1) as r1:
        v1 = await handler1(**r1)
    async with resolved_dependencies(handler2) as r2:
        v2 = await handler2(**r2)

assert v1 == v2 == 1 and init_count == 1, (v1, v2, init_count)
print(f"shared id reused: v1={v1} v2={v2} init_count={init_count}")

In [ ]:
# Example 5: FailedDependency + validate_dependencies + kwargs override.
from uncalled_for import Depends, FailedDependency, resolved_dependencies, validate_dependencies

def good_factory() -> int:
    return 100

def broken_factory() -> int:
    raise RuntimeError("boom")

async def target(good: int = Depends(good_factory), bad: int = Depends(broken_factory)) -> int:
    return good

validate_dependencies(target)

async with resolved_dependencies(target) as resolved:
    assert resolved["good"] == 100
    assert isinstance(resolved["bad"], FailedDependency)
    assert isinstance(resolved["bad"].error, RuntimeError)

async with resolved_dependencies(target, kwargs={"bad": 999}) as resolved:
    assert resolved["bad"] == 999

print("FailedDependency + override + validate_dependencies OK")
assert True